# Extending Jaya as a Modern Hyperparameter Optimization Engine

This repository extends the Jaya algorithm into a practical Hyperparameter Optimization (HPO) engine supporting:

Continuous parameters, Integer parameters, Categorical parameters, Mixed-type search spaces

Below are minimal working implementations for Easy, Medium, and Hard levels.

1. Easy Level – Integer Optimization
Problem

Optimize max_depth of a Decision Tree (Iris dataset) using 5-fold CV accuracy.

Solution 1: Jaya-Based Optimization

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import cross_val_score

X, y = load_iris(return_X_y=True)

def objective(depth):
    depth = int(round(depth))
    depth = max(1, min(depth, 20))
    model = DecisionTreeClassifier(max_depth=depth)
    score = cross_val_score(model, X, y, cv=5).mean()
    return -score  # minimize

def jaya_optimize(pop_size=12, iterations=25):
    population = np.random.randint(1, 21, pop_size)

    for _ in range(iterations):
        fitness = np.array([objective(p) for p in population])
        best = population[np.argmin(fitness)]
        worst = population[np.argmax(fitness)]

        new_population = []
        for p in population:
            new_value = p \
                + np.random.rand() * (best - abs(p)) \
                - np.random.rand() * (worst - abs(p))

            new_population.append(int(np.clip(new_value, 1, 20)))

        population = np.array(new_population)

    return best

print("Best Depth (Jaya):", jaya_optimize())

Best Depth (Jaya): 8


Behavior:
This implementation applies Jaya’s update rule to integer search space using rounding and boundary control. It iteratively improves solutions by moving toward the best and away from the worst candidate

Solution 2: Random Search Baseline

In [ ]:
import random

def random_search():
    best_score = 0
    best_depth = 1

    for _ in range(20):
        depth = random.randint(1, 20)
        model = DecisionTreeClassifier(max_depth=depth)
        score = cross_val_score(model, X, y, cv=5).mean()

        if score > best_score:
            best_score = score
            best_depth = depth

    return best_depth

print("Best Depth (Random):", random_search())

Best Depth (Random): 3


Behavior:
This method randomly samples depth values within bounds and keeps the best observed solution. It serves as a simple baseline without guided search.

# 2. Medium Level – Mixed-Type Optimization
Problem

Random Forest tuning:

n_estimators (Integer)

max_depth (Integer)

criterion (Categorical: "gini", "entropy")

Solution 1: Continuous-to-Discrete Mapping

In [ ]:
from sklearn.ensemble import RandomForestClassifier

def decode_solution(x):
    n_estimators = int(round(50 + x[0] * 150))
    max_depth = int(round(2 + x[1] * 18))
    criterion = ["gini", "entropy"][int(x[2] * 2) % 2]
    return n_estimators, max_depth, criterion

def rf_objective(x):
    n_estimators, max_depth, criterion = decode_solution(x)
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        criterion=criterion
    )
    score = cross_val_score(model, X, y, cv=5).mean()
    return -score

Behavior:
The optimizer works in normalized continuous space and decodes values into valid ML hyperparameters. This allows Jaya to operate without modification while supporting mixed parameter types.

Solution 2: Embedded Categorical Encoding

In [ ]:
def decode_embedded(x):
    n_estimators = int(round(50 + x[0] * 150))
    max_depth = int(round(2 + x[1] * 18))

    idx = np.argmax([x[2], x[3]])
    criterion = ["gini", "entropy"][idx]

    return n_estimators, max_depth, criterion

Behavior:
Categorical choices compete through multi-dimensional representation. The category with highest encoded value is selected, improving stability in discrete transitions.

# 3. Hard Level – Generic Jaya Engine

Solution 1: Functional Design

In [ ]:
def jaya_tune(eval_fn, bounds, pop_size=15, iterations=30):
    dim = len(bounds)
    population = np.random.rand(pop_size, dim)

    for d in range(dim):
        low, high = bounds[d]
        population[:, d] = low + population[:, d] * (high - low)

    for _ in range(iterations):
        fitness = np.array([eval_fn(ind) for ind in population])
        best = population[np.argmin(fitness)]
        worst = population[np.argmax(fitness)]

        for i in range(pop_size):
            candidate = population[i] \
                + np.random.rand(dim) * (best - abs(population[i])) \
                - np.random.rand(dim) * (worst - abs(population[i]))

            for d in range(dim):
                low, high = bounds[d]
                candidate[d] = np.clip(candidate[d], low, high)

            population[i] = candidate

    return best

Behavior:
This is a generic optimizer that accepts any evaluation function and bounded search space. It can handle continuous problems directly and mixed-type problems through decoding.

Solution 2: Object-Oriented Design

In [ ]:
def jaya_tune(eval_fn, bounds, pop_size=15, iterations=20):
    dim = len(bounds)
    population = np.random.rand(pop_size, dim)

    for d in range(dim):
        low, high = bounds[d]
        population[:, d] = low + population[:, d] * (high - low)

    best_score = float("inf")
    best_solution = None

    for iteration in range(iterations):
        fitness = np.array([eval_fn(ind) for ind in population])

        current_best_idx = np.argmin(fitness)
        current_worst_idx = np.argmax(fitness)

        best = population[current_best_idx]
        worst = population[current_worst_idx]

        if fitness[current_best_idx] < best_score:
            best_score = fitness[current_best_idx]
            best_solution = best

        print(f"Iteration {iteration+1} | Best Score: {-best_score:.4f}")

        for i in range(pop_size):
            candidate = population[i] \
                + np.random.rand(dim) * (best - abs(population[i])) \
                - np.random.rand(dim) * (worst - abs(population[i]))

            for d in range(dim):
                low, high = bounds[d]
                candidate[d] = np.clip(candidate[d], low, high)

            population[i] = candidate

    return best_solution

In [ ]:
best_solution = jaya_tune(rf_objective, bounds)

print("\nBest Encoded Vector:", best_solution)
print("Decoded Parameters:", decode_solution(best_solution))
print("Final Accuracy:", -rf_objective(best_solution))

Iteration 1 | Best Score: 0.9667
Iteration 2 | Best Score: 0.9667
Iteration 3 | Best Score: 0.9667
Iteration 4 | Best Score: 0.9667
Iteration 5 | Best Score: 0.9667
Iteration 6 | Best Score: 0.9667
Iteration 7 | Best Score: 0.9667
Iteration 8 | Best Score: 0.9667
Iteration 9 | Best Score: 0.9667
Iteration 10 | Best Score: 0.9667
Iteration 11 | Best Score: 0.9667
Iteration 12 | Best Score: 0.9667


KeyboardInterrupt: 

Behavior:
This class-based implementation encapsulates Jaya into a reusable optimizer object. It improves modularity and makes future extensions like restarts or logging easier.

#Why You May Not See Strong Visible Output
 Jaya Functions Only Return Values (They Don’t Print Progress)
 They return the best solution internally, but they do not print intermediate results like:

Current iteration

Current best score

Improvement over time

So it looks silent, but it is actually working.